# Ensemble Learning Techniques: From First Principles

## 1. Why Combine Models?

A Decision Tree may follow accidental details in the training sample. A linear model may miss a curved relationship. A nearest-neighbor model may be sensitive to the particular examples around a new observation. These methods can make different predictions because they represent the data differently.

**Ensemble learning** combines several fitted models into one prediction procedure. The hope is that useful signals reinforce each other while some errors cancel. Asking several experts is a helpful analogy, but agreement is only valuable if their evidence is sound. Ten models repeating the same mistake do not make that mistake more reliable.

For a running classification example, imagine predicting whether a manufactured part is defective. We might combine a linear classifier, a tree, and a neighbor-based classifier. For regression, we could combine models that estimate the part's lifetime.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Combine useful models whose errors differ enough to make the combined prediction better. More models alone do not guarantee an improvement.</div>

This lesson covers voting, averaging, bagging, boosting, stacking, and blending. These describe different ways to train or combine models; they are not interchangeable names for the same algorithm.

## 2. What Goes Into an Ensemble?

An ensemble receives the same kind of input as the task it solves: a feature row for classification or regression. Internally, its component models transform that row into predictions, which are combined into a final output.

Let $M$ be the number of models, $\mathbf x$ a feature vector, and $f_m(\mathbf x)$ the prediction from model $m$. A classifier may instead supply class label $\hat c_m(\mathbf x)$ or class probability $p_{mk}(\mathbf x)$ for class $k$.

| Term | Meaning |
|---|---|
| Base learner | The learning method used for a component model |
| Base estimator | One fitted component of an ensemble |
| Homogeneous ensemble | Components use the same model family, such as many trees |
| Heterogeneous ensemble | Components use different model families |
| Aggregation | The rule combining component predictions |
| Diversity | Differences in predictions or errors across components |
| Meta-model | A learned model that combines base-model outputs |

The base estimators need not all be weak. Voting can combine individually strong models; boosting often uses deliberately restricted learners. Diversity can arise from different algorithms, data samples, features, randomization, or training objectives. Different model names do not by themselves establish useful diversity.

## 3. Bias, Variance, and Correlated Errors

**Bias** is systematic error in the average prediction across repeated training samples. **Variance** describes how predictions fluctuate across those samples. Noise not explained by the available inputs remains another source of prediction error. The familiar bias–variance decomposition is most direct for squared-error regression; classification requires more care.

Averaging can reduce variation when the component predictions do not fluctuate together perfectly. At a fixed input, suppose each model's prediction has variance $\sigma^2$ and every pair has correlation $\rho$. The equal-weight average has variance

$$
\operatorname{Var}(\bar f)=\frac{1}{M^2}\left[M\sigma^2+M(M-1)\rho\sigma^2\right]
=\sigma^2\left[\rho+\frac{1-\rho}{M}\right].
$$

The first term inside the brackets accounts for individual variances; the second counts covariance between different models. This is an illustrative equal-variance, equal-correlation calculation, not a universal description of every ensemble.

With $M=10$, $\sigma^2=4$, and $\rho=0.5$, the average's variance is $4(0.5+0.5/10)=2.2$. With uncorrelated fluctuations it would be $4/10=0.4$; with perfect correlation it stays 4. More models cannot eliminate their shared error component.

Averaging also does not automatically remove shared bias. If every lifetime model systematically predicts too high, their mean can remain too high. The useful tradeoff is component quality together with complementary errors, not maximum disagreement at any cost.

## 4. Hard Voting: Combine Class Labels

Hard voting gives each model a vote for its predicted class. For $K$ classes, the equal-weight rule is

$$
\hat c(\mathbf x)=\arg\max_{k\in\{1,\ldots,K\}}
\sum_{m=1}^{M}\mathbb{1}[\hat c_m(\mathbf x)=k].
$$

The indicator $\mathbb{1}[\cdot]$ equals one when the statement is true and zero otherwise. The operation $\arg\max$ chooses the class receiving the largest count. In a multiclass task, the winning class can have the most votes without receiving more than half of them.

For a part inspected by three models:

| Model | Prediction |
|---|---|
| Linear classifier | Defective |
| Tree | Acceptable |
| Neighbor classifier | Defective |

Defective wins by two votes to one. We do not yet know whether that decision is correct because the true outcome has not been supplied.

Weighted voting assigns a nonnegative weight $a_m$ to each model's indicator instead of counting every vote equally. Weights should be chosen using development data, not the final test results. Ties need a defined implementation rule; an odd model count avoids some binary ties but not every multiclass or weighted tie.

## 5. Soft Voting: Combine Class Probabilities

Soft voting averages probabilities for each class, then selects the largest average. With nonnegative weights and a positive weight sum,

$$
\bar p_k(\mathbf x)=\frac{\sum_{m=1}^{M}a_mp_{mk}(\mathbf x)}{\sum_{m=1}^{M}a_m},
\qquad \hat c(\mathbf x)=\arg\max_k\bar p_k(\mathbf x).
$$

Every probability column must refer to the same class. These are probability estimates, not arbitrary decision scores: averaging an SVM margin with a probability would mix incompatible scales.

Consider the same three models, now with hypothetical probabilities of defect:

| Model | Probability of defect | Class at threshold 0.5 |
|---|---:|---|
| Linear classifier | 0.60 | Defective |
| Tree | 0.05 | Acceptable |
| Neighbor classifier | 0.60 | Defective |

Hard voting predicts defective. Equal-weight soft voting gives $(0.60+0.05+0.60)/3=1.25/3\approx0.4167$, so it predicts acceptable. With weights $(2,1,2)$, the probability becomes $(1.20+0.05+1.20)/5=0.49$, still below 0.5.

The tree's strong probability estimate influences the average more than its one hard vote. This helps only if its probability scale is informative. An overconfident, poorly calibrated component can harm the ensemble. Soft voting is therefore not guaranteed to outperform hard voting, and averaging calibrated models does not automatically prove the mixture is calibrated.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>Hard and soft voting answer different aggregation questions and can disagree on the same row. Evaluate both using unseen observations; confidence alone does not establish correctness.</div>

## 6. Regression Averaging and a Numerical Check

For numerical predictions, the equal-weight ensemble is

$$
\hat y(\mathbf x)=\frac1M\sum_{m=1}^{M}f_m(\mathbf x).
$$

If three models predict lifetimes of 90, 100, and 110 hours, their average is 100 hours. With weights $(1,2,1)$ it is also $(90+200+110)/4=100$. If the actual lifetime is 102, their individual squared errors are 144, 4, and 64; the average prediction has squared error 4.

For equal weights, the squared error of the mean cannot exceed the mean of the individual squared errors on that same row. That mathematical fact does not mean it beats the best individual model. If the truth were 90, the first model would have zero error while the average would have squared error 100.

A poor added model can therefore make an existing ensemble worse. Validate membership and weights, and compare with the strongest individual baseline. Nonnegative normalized weights keep a weighted prediction within the range of component predictions; an unrestricted learned combiner need not obey that bound.

## 7. Bagging: Create Models Through Resampling

Bagging means **bootstrap aggregating**. It typically builds many instances of the same learner on different samples drawn with replacement from the training data, then combines their predictions.

1. Start with $n$ development training rows.
2. Draw a bootstrap sample, commonly containing $n$ draws with replacement.
3. Fit a base estimator to that sample.
4. Repeat with new draws for each estimator.
5. Aggregate numerical predictions or class information when making predictions.

For rows A, B, C, D, E, one bootstrap draw might be A, C, C, E, A. Repetition is intentional. B and D are omitted from this particular draw, but other models can use them. Bootstrap samples are different, not disjoint.

Bagging is especially useful for unstable learners such as deep trees. A changed sample can produce a different tree, and averaging can reduce that instability. Bagging an already stable model may produce little gain. Independent fitting jobs also do not mean statistically independent predictions: the models share the original data source and often similar patterns.

For classification, aggregation details depend on the implementation. Scikit-learn's `BaggingClassifier` averages class probabilities when its base estimator supplies them; otherwise it can aggregate votes. A tree bagger therefore need not behave like a majority vote over tree labels. See the [ensemble guide](https://scikit-learn.org/stable/modules/ensemble.html).

## 8. Out-of-Bag Evaluation and Sampling Variants

In $n$ independent draws with replacement from $n$ rows, a particular row is omitted with probability

$$
P(\text{omitted})=\left(1-\frac1n\right)^n\longrightarrow e^{-1}\approx0.368.
$$

Each draw misses the row with probability $1-1/n$, and multiplying over $n$ draws gives the expression. The limiting value applies approximately for large $n$. The expected fraction of distinct included rows is therefore about 0.632, not 1, despite making $n$ draws.

For five rows, the omission probability is $(4/5)^5=0.32768$. A realized bootstrap sample can omit more or fewer rows than its expectation.

**Out-of-bag (OOB) prediction** for one training row aggregates only models whose bootstrap samples omitted that row. Comparing these predictions with known targets provides an internal evaluation estimate. It requires enough applicable models per row and does not automatically accommodate time dependence or grouped observations. Repeatedly tuning against OOB scores can also overfit that evaluation signal.

| Method | Main randomization |
|---|---|
| Bagging | Samples rows with replacement |
| Pasting | Samples rows without replacement |
| Random subspace | Uses subsets of features for base models |
| Random patches | Samples both rows and features |

A pipeline fitted globally before bootstrapping can expose preprocessing information from a nominally OOB row. Where strict exclusion matters, the learned preprocessing must be included in the resampled training procedure too. OOB is a useful tool, not a universal substitute for an independent assessment.

## 9. Where Random Forest Fits

A typical Random Forest builds trees on bootstrap samples and restricts the candidate features considered at each split. Feature sampling reduces the chance that every tree uses the same dominant predictor at the top and repeats the same mistakes.

This differs from choosing one fixed feature subset for an entire tree in the random-subspace method. Random Forest usually redraws candidate features at successive splits. Reducing the candidate set can make trees less correlated but can also weaken individual trees. The validation task is to find a useful balance.

Regression forests average tree predictions. Scikit-learn's classification forest averages tree class probabilities and takes the class with the largest average, which can differ from hard voting over labels. This implementation distinction matters when interpreting a forest as “many votes.”

More trees generally stabilize the ensemble's random approximation, but gains can diminish as shared errors dominate. Tree depth, leaf size, and feature sampling still control what the components learn. A large forest does not automatically repair leakage, missing signal, or a biased target definition.

The separate Random Forest notes develop these mechanics in depth; here it serves as the bridge between resampling, feature randomization, and aggregation.

## 10. Boosting: Build an Additive Model Sequentially

Boosting builds models in sequence, with later stages responding to the current ensemble. A general additive form is

$$
F_M(\mathbf x)=F_0(\mathbf x)+\sum_{m=1}^{M}\eta a_mh_m(\mathbf x).
$$

Here $F_0$ is an initial prediction or score, $h_m$ is the stage-$m$ learner, $a_m$ is its stage weight when used, and $\eta>0$ is a learning-rate factor. Exact conventions differ between boosting algorithms. In classification, $F_M$ may be a score transformed into a probability or label, rather than a direct average of probabilities.

AdaBoost changes observation weights to emphasize examples handled poorly and combines learners with learned weights. Gradient boosting fits updates related to the negative gradient of a chosen loss. For squared-error regression, those targets are the residuals; for other losses, “fit the residuals” is not generally the full description.

For one squared-error regression row with actual target 10 and current prediction 6, the residual is 4. If the next learner predicts an update of 3 and the learning rate is 0.2, the new prediction is $6+0.2(3)=6.6$. Its residual becomes 3.4 and squared error falls from 16 to 11.56. The learner fits a pattern across rows, not a separately memorized correction for each row.

This example shows the additive update, not a guarantee that every validation row improves at every stage. Learning rate, number of stages, tree complexity, and stopping rules interact. More stages can eventually overfit, especially when the procedure chases noise.

## 11. Compare Bagging and Boosting

| Dimension | Bagging | Boosting |
|---|---|---|
| Training relationship | Base fits can run separately | Later stages depend on the current ensemble |
| Source of variation | Resampled rows, possibly feature randomization | Changing weights, gradients, or additive fitting targets |
| Typical aggregation | Averaging probabilities or values, or voting | Algorithm-specific weighted/additive score |
| Common benefit | Reduced variance of unstable learners | Improved approximation through sequential corrections |
| Important controls | Number of models, sample size, base complexity | Number of stages, learning rate, base complexity |
| Main caution | Correlated errors limit averaging gains | Noise and aggressive fitting can harm generalization |

The summary “bagging reduces variance, boosting reduces bias” is useful as an introduction, but not a strict division. Both can affect multiple aspects of prediction error. Neither is universally better.

Random Forest belongs to the randomized averaging family. AdaBoost and Gradient Boosting belong to boosting. XGBoost, LightGBM, and CatBoost are gradient-boosting systems with distinct optimization and representation choices; their separate notebooks explain those differences. They should not be treated as identical implementations or as automatic upgrades in every setting.

Even when stages must be trained sequentially, work inside a boosting stage may be parallelized. Conversely, independently trainable models still compete for memory and computational resources.

## 12. Stacking: Learn the Combination Without Leakage

Stacking uses predictions from base models as features for a meta-model. For example, each row might receive three defect probabilities from a linear classifier, KNN, and a tree. A final Logistic Regression model learns how to combine those outputs.

The critical requirement is that meta-training predictions come from base models that did **not** train on the corresponding row. Otherwise, an overfitting base model can appear unrealistically reliable to the meta-model.

A cross-validated construction is:

1. Reserve the final test set before building the stack.
2. Divide development training rows into folds appropriate to the data.
3. For each fold, fit every base pipeline on the other folds and predict that held-out fold.
4. Assemble these out-of-fold predictions into a meta-feature table, with one row per development observation.
5. Fit the meta-model using that table and the development labels.
6. Refit base pipelines on all development data for prediction on new rows. Pass their outputs to the fitted meta-model.

For six illustrative rows A–F and three folds, predict A–B using base fits trained on C–F; predict C–D using fits trained on A–B and E–F; predict E–F using fits trained on A–D. Every meta-feature row is generated without its own training label entering that base fit.

| Meta-training row | Base model 1 probability | Base model 2 probability | Target |
|---|---:|---:|---|
| A | 0.70 | 0.40 | Defective |
| B | 0.20 | 0.10 | Acceptable |

These are illustrative outputs, not fitted results. The meta-model may learn relative trust patterns, but its ability depends on its own structure and inputs. A simple linear combiner does not automatically learn every context-specific rule.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Stacking pitfall</strong><br>Do not train the meta-model on predictions made by base models fitted on those same rows. Preprocessing belongs inside the base folds too, and the complete stack still needs independent evaluation.</div>

Out-of-fold construction trains the stack; it is not by itself an unbiased final score for a meta-model fitted on those outputs. Use a held-out test set or an outer validation loop to evaluate the complete procedure. If base hyperparameters are selected using all development labels, fully strict nested estimates require that selection to occur within the appropriate training folds as well.

## 13. Blending and Choosing a Combining Strategy

Blending commonly means fitting base models on one development subset and training a combiner on predictions for a separate holdout subset. The combiner learns from labels the base fits did not see. This is simpler than producing out-of-fold predictions, but leaves fewer rows for fitting each stage and can depend strongly on one split.

Do not call the blending holdout the final test set: its labels train the combiner. A further independent evaluation is required. If base models are later refitted on a larger dataset, their prediction distributions may change relative to the outputs used to train the blender; evaluate that full deployment procedure rather than assuming the refit is harmless.

| Combining method | What is learned beyond base models? | Typical concern |
|---|---|---|
| Equal voting or averaging | No additional fitted combination weights | Some components may dilute stronger models |
| Weighted voting or averaging | Chosen or fitted weights | Weights can overfit validation results |
| Stacking | A meta-model using out-of-fold features | More fitting and careful nested data separation |
| Blending | A meta-model using a holdout subset | Reduced data for each stage and split sensitivity |

Start with the simplest combination that provides a measurable benefit. Stacking is not inherently superior to a well-tuned single model or an equal-weight average.

## 14. A Practical Training and Evaluation Workflow

1. Establish individual baselines and define the primary metric, label orientation, and relevant error costs.
2. Reserve an independent test set. Respect groups, time order, and near-duplicates when choosing splits.
3. Fit model-specific preprocessing inside pipelines: scale distance-based and appropriate linear models; use suitable encoding and missing-value handling for each learner.
4. Compare candidate ensemble designs using development-only validation. Tune base complexity, combination choices, and relevant ensemble settings together.
5. Inspect complementary errors on validation predictions, not only individual accuracy. Consider latency, memory, and stability across folds.
6. Select the full procedure, refit using development data, and evaluate once on the reserved test set.
7. Apply the entire fitted procedure consistently to future observations and monitor distribution changes.

| Setting | Effect and tradeoff |
|---|---|
| Number of estimators | More capacity or averaging stability, with higher cost; behavior differs by family |
| Base-model complexity | Controls what each component can learn and how much it can overfit |
| Row/feature sampling | Trades individual information against diversity |
| Voting weights | Changes influence of each component; requires validation |
| Boosting learning rate | Scales sequential updates and interacts with stage count |
| Meta-model complexity | Controls flexibility and overfitting of the learned combination |

Scaling is not a universal ensemble requirement; it follows the component model. A heterogeneous ensemble can contain different preprocessing pipelines. Training one global scaler before validation would still leak fold information even if the final combination is sophisticated.

## 15. Connect the Concepts to the Companion

`13_Ensemble_Learning_Techniques.ipynb` uses `load_breast_cancer`, an 80/20 split, seed 42, and stratification. It constructs Logistic Regression and KNN pipelines with `StandardScaler`, plus a depth-4 Decision Tree, then compares:

| Companion model | Configuration shown |
|---|---|
| Hard voting | Equal votes from Logistic Regression, KNN with 7 neighbors, and the tree |
| Soft voting | Equal probability averaging from the same model specifications |
| Bagging | 100 Decision Trees with default bootstrap sampling |
| Random Forest | 200 trees, `max_features="sqrt"` |
| AdaBoost | Up to 100 depth-1 trees, `learning_rate=0.5` |

A depth-1 tree is a **decision stump**, with one split. AdaBoost's configured estimator count can be an upper limit because fitting may stop early. The comparison does not implement stacking, blending, or OOB evaluation, and it does not tune these settings.

The dataset labels are 0 for malignant and 1 for benign. Consequently, default binary F1 in this companion treats benign as positive, and probability column `[:, 1]` supplies benign scores for ROC-AUC. If the question concerns malignant detection, choose that positive event explicitly and inspect its recall and errors.

The results table contains no ROC-AUC value for hard voting or bagging because the code does not calculate those entries. Hard voting supplies labels rather than soft probabilities; the bagger with tree estimators can supply probabilities, so its missing table entry is not a general limitation of bagging.

The code uses a common test split, which aligns evaluated rows, but equal access to a split does not make the models equally tuned or computationally matched. Choosing repeated changes by inspecting this table would use the test set for model selection.

These notes describe the source code. They do not claim measured scores, fitted tree counts, or a winning ensemble because the companion was not executed for this revision.

## 16. Metrics, Interpretation, and Practical Limits

For classification, inspect the confusion matrix and per-class performance. With true positives TP, false positives FP, and false negatives FN for the chosen positive class,

$$
\mathrm{Precision}=\frac{TP}{TP+FP},\qquad
\mathrm{Recall}=\frac{TP}{TP+FN},\qquad
F_1=\frac{2TP}{2TP+FP+FN}.
$$

Use a defined policy for zero denominators. Accuracy summarizes overall correctness but can hide minority-class failures. ROC-AUC evaluates ranking; calibration evaluates whether predicted probabilities match frequencies. Soft voting does not remove the need to assess calibration and choose an appropriate operating threshold.

For regression with $N$ evaluation rows and errors $e_i=y_i-\hat y_i$,

$$
\mathrm{MAE}=\frac1N\sum_i|e_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac1N\sum_i e_i^2}.
$$

Both use target units, but RMSE gives large errors more influence. Compare the ensemble and its strongest component on the same rows. Inspect relevant subgroups and periods rather than relying only on an aggregate improvement.

Component disagreement can be informative, but it is not automatically a calibrated uncertainty interval. Models can agree because they share a blind spot. Feature importances, voting weights, and meta-model coefficients describe predictive behavior, not causal effects; correlated inputs and correlated model predictions complicate interpretation.

Prediction usually requires evaluating multiple components. For similarly sized models, work can grow roughly with the number evaluated; parallel execution changes elapsed time but does not erase memory or computation costs. Stacking adds the combiner's cost and cross-validation multiplies training fits. Maintain all preprocessing, label mappings, and fitted components needed by the deployed procedure.

## 17. Failure Modes and Recap

| Failure mode | Why combination does not fix it | Practical response |
|---|---|---|
| Shared systematic mistakes | Averaging preserves common bias | Improve data or representation and inspect error overlap |
| Weak added components | Diversity can come from poor predictions | Validate the actual combination against strong baselines |
| Overconfident probabilities | Soft voting gives their numbers influence | Assess calibration and alternative aggregation |
| Leakage in stacking or preprocessing | Artificially good meta-features mislead the learner | Generate predictions with proper fold separation |
| Noisy targets or distribution shift | All components can learn the same unsuitable signal | Audit data and use realistic evaluation |
| Excessive complexity | Small score gains may cost too much | Compare latency, memory, maintainability, and uncertainty |

Ensembles are useful when their combination improves validated predictions enough to justify the cost. A simple model may still be preferable when it performs similarly, is easier to inspect, or satisfies operational constraints better. No ensemble can create missing information merely by increasing the number of learners.

The central choices are how to create complementary models and how to combine their outputs. Voting combines labels or probabilities; regression averaging combines numbers; bagging resamples training data; boosting builds sequential updates; stacking and blending learn a combination from held-out predictions.

Remember the two key cautions: shared errors limit the benefit of more models, and every learned choice—including a combiner—must respect the training/validation boundary. A well-validated combination is the goal, not the largest collection of models.